In [11]:
!wget https://data.ncl.ac.uk/ndownloader/articles/24551539/versions/1 -O ../data/CIFARTile.zip
!mkdir ../data/CIFARTile
!unzip ../data/CIFARTile.zip -d ../data/CIFARTile/
!rm ../data/CIFARTile.zip

--2025-05-11 20:02:02--  https://data.ncl.ac.uk/ndownloader/articles/24551539/versions/1
Resolving data.ncl.ac.uk (data.ncl.ac.uk)... 34.255.146.38, 52.48.143.184, 54.216.15.6
Connecting to data.ncl.ac.uk (data.ncl.ac.uk)|34.255.146.38|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3441203334 (3.2G) [application/zip]
Saving to: ‘../data/CIFARTile.zip’

../data/CIFARTile.z 100%[===================>]   3.20G  22.0MB/s    in 3m 8s   

2025-05-11 20:05:10 (17.5 MB/s) - ‘../data/CIFARTile.zip’ saved [3441203334/3441203334]

Archive:  ../data/CIFARTile.zip
 extracting: ../data/CIFARTile/train_y.npy  
 extracting: ../data/CIFARTile/test_y.npy  
 extracting: ../data/CIFARTile/valid_y.npy  
 extracting: ../data/CIFARTile/valid_x.npy  
 extracting: ../data/CIFARTile/test_x.npy  
 extracting: ../data/CIFARTile/train_x.npy  
 extracting: ../data/CIFARTile/metadata  
 extracting: ../data/CIFARTile/README.txt  


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from senmodel.model.utils import *
from senmodel.metrics.nonlinearity_metrics import *
from senmodel.metrics.edge_finder import *
from senmodel.metrics.train_metrics import *
from senmodel.train.train import *

In [2]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=3 * 64 * 64, output_size=4):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [3]:
hyperparams = {
    "num_epochs": 32,
    "batch_size": 256,
    "metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "aggregation_mode": "mean",
    "choose_thresholds": {"fc0": 0.1}, # 1.0 -> no edges, 0.0 -> all edges
    "choose_thresholds_del": {"fc0": 0.1}, # 1.0 -> no edges, 0.0 -> all edges
    "threshold": 0.05,
    "min_delta_epoch_replace": 16,
    "window_size": 5,
    "lr": 1e-5,
    "delete_after": 2,    
    "task_type": "classification",
    "fully_connected": False,
    "max_to_replace": 1000, # None -> no limit
    "weight_decay": 1e-2,
}

name = ", ".join(
    f"{key}: {value.__class__.__name__ if key == 'metric' else value}"
    for key, value in hyperparams.items()
)

name

"num_epochs: 32, batch_size: 256, metric: AbsGradientEdgeMetric, aggregation_mode: mean, choose_thresholds: {'fc0': 0.1}, choose_thresholds_del: {'fc0': 0.99}, threshold: 0.05, min_delta_epoch_replace: 16, window_size: 5, lr: 1e-05, delete_after: 2, task_type: classification, fully_connected: False, max_to_replace: 1000, weight_decay: 0.01"

In [4]:
train_x = np.load("../data/CIFARTile/train_x.npy")
train_y = np.load("../data/CIFARTile/train_y.npy")
valid_x = np.load("../data/CIFARTile/valid_x.npy")
valid_y = np.load("../data/CIFARTile/valid_y.npy")
test_x = np.load('../data/CIFARTile/test_x.npy')
test_y = np.load('../data/CIFARTile/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 3 * 64 * 64)
train_tensor_y = torch.tensor(train_y)

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 3 * 64 * 64)
valid_tensor_y = torch.tensor(valid_y)

test_tensor_x = torch.tensor(test_x).float().view(-1, 3 * 64 * 64)
test_tensor_y = torch.tensor(test_y)


train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)


train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False)

In [5]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [6]:
import wandb

wandb.login()
wandb.finish()
run = wandb.init(
    project="self-expanding-nets-CIFARTile",
    name=f"trash",
    config=hyperparams
)

wandb: Currently logged in as: vanyamironov to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [8]:
import time
start_time = time.time()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['lr'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, train_loader, val_loader, criterion, optimizer, hyperparams, device)

print("--- %s seconds ---" % (time.time() - start_time))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:02<00:00, 70.99it/s]


Epoch 1/32, Train Loss: 1.4529, Val Loss: 1.4284, Val Accuracy: 0.2500


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:02<00:00, 80.85it/s]


Epoch 2/32, Train Loss: 1.4194, Val Loss: 1.4174, Val Accuracy: 0.2508


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:02<00:00, 79.39it/s]


Epoch 3/32, Train Loss: 1.4094, Val Loss: 1.4126, Val Accuracy: 0.2509


... (54 outputs left out)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:03<00:00, 50.69it/s]


Epoch 31/32, Train Loss: 1.3576, Val Loss: 1.4102, Val Accuracy: 0.2493


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:03<00:00, 50.34it/s]


Epoch 32/32, Train Loss: 1.3570, Val Loss: 1.4109, Val Accuracy: 0.2499
--- 343.7535967826843 seconds ---


In [9]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [10]:
accuracy, params

(0.2598, 615152)

In [ ]:
torch.save(sparse_model, 'cifartile.pt')

In [11]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 45.56, 48.88, 24.43, 31.06, 48.49, 51.28, 46.23, 92.28, 81.08, 91.47, 90.74, 24.76, 25],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))
styled_table

Model,Accuracy (%),Parameters
Ours,25.98,"615,152.0"
ResNet-18,45.56,"11,689,512.0"
AlexNet,48.88,"61,100,840.0"
VGG16,24.43,"138,357,544.0"
ConvNext,31.06,"88,591,464.0"
MNASNet,48.49,"4,383,312.0"
DenseNet,51.28,"28,681,000.0"
ResNeXt,46.23,"25,028,904.0"
PC-DARTS,92.28,nan
DrNAS,81.08,nan
